# Plot All Satellite Trail Pipelines

Render the original image, true mask, base model outputs, and every final-evaluation postprocessing pipeline in one figure per image. Only the final matplotlib figures are saved to Google Drive; intermediate prediction masks stay in memory for the current image and are cleared after plotting.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Navigate to your Drive
%cd /content/drive/MyDrive

# Move into the repository folder
%cd satellite_trails

In [ ]:
!pip install -e .[dev]

## Configuration

In [ ]:
import csv
from pathlib import Path
import re
import sys

import matplotlib.pyplot as plt
import numpy as np
import torch
from PIL import Image

Image.MAX_IMAGE_PIXELS = 150000000

REPO_ROOT = Path.cwd()
SRC_ROOT = REPO_ROOT / "src"
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

from segmentation import SatelliteTrailPipeline
from satellite_trail_segmentation.classifier_model.classifier import TrailClassifier
from satellite_trail_segmentation.ml_utils.checkpoints import load_checkpoint
from satellite_trail_segmentation.postprocess.pipeline import postprocess_segmentation
from satellite_trail_segmentation.unet_model.unet import UNet

DATA_DIR = REPO_ROOT / "data" / "test_images"
if not DATA_DIR.exists():
    DATA_DIR = Path("/content/test_images")
MASTER_SPLIT_CSV = REPO_ROOT / "data" / "h5s" / "master_split.csv"
MODEL_CHECKPOINT_DIR = Path("/content/drive/MyDrive/satellite_trails/results/models")
OUTPUT_DIR = Path("/content/drive/MyDrive/satellite_trails/results/pipeline_plots")

UNET_CHECKPOINT = MODEL_CHECKPOINT_DIR / "unet" / "unet_weights.pt"
CLASSIFIER_CHECKPOINT = MODEL_CHECKPOINT_DIR / "classifier" / "classifier_weights.pt"

SELECTED_IMAGE_NAMES = []
SPLIT_ID = "2"
NORMALIZATION = "source_zscore"
PATCH_DIM = 528
UNET_THRESHOLD = 0.65
CLASSIFIER_THRESHOLD = 0.725
UNET_BATCH_SIZE = None
CLASSIFIER_BATCH_SIZE = None
NUM_WORKERS = 2
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Using device: {DEVICE}")
print(f"DATA_DIR: {DATA_DIR}")
print(f"OUTPUT_DIR: {OUTPUT_DIR}")

## Model And Pipeline Setup

In [ ]:
def build_model(model_class, checkpoint_path, allowed_keys):
    checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=False)
    model_config = checkpoint.get("model_config", {}) or {}
    model_kwargs = {key: model_config[key] for key in allowed_keys if key in model_config}
    model = model_class(**model_kwargs)
    load_checkpoint(checkpoint_path, model)
    model.eval()
    return model


unet_model = build_model(
    UNet,
    UNET_CHECKPOINT,
    ("in_channels", "out_channels", "kernel_size", "base_channels", "dropout", "use_batchnorm"),
)
classifier_model = build_model(
    TrailClassifier,
    CLASSIFIER_CHECKPOINT,
    ("in_channels", "kernel_size", "base_channels", "dropout"),
)

pipeline = SatelliteTrailPipeline(
    unet_model,
    classifier_model=classifier_model,
    patch_dim=PATCH_DIM,
    device=DEVICE,
    unet_batch_size=UNET_BATCH_SIZE,
    classifier_batch_size=CLASSIFIER_BATCH_SIZE,
    num_workers=NUM_WORKERS,
)

POSTPROCESS_METHODS = {
    "postprocess_asta_none": {
        "hough_threshold": 50,
        "min_line_length": 100,
        "max_line_gap": 250,
        "morph_kernel_size": 3,
        "min_component_size": 500,
        "contour_filter": True,
        "contour_area_threshold": 3000,
        "line_mode": "asta",
        "width_mode": "none",
    },
    "postprocess_asta_median_sampled_width": {
        "hough_threshold": 50,
        "min_line_length": 100,
        "max_line_gap": 250,
        "morph_kernel_size": 3,
        "min_component_size": 500,
        "contour_filter": True,
        "contour_area_threshold": 3000,
        "line_mode": "asta",
        "width_mode": "median_sampled_width",
    },
    "postprocess_centerline_median_sampled_width": {
        "hough_threshold": 50,
        "min_line_length": 100,
        "max_line_gap": 250,
        "morph_kernel_size": 3,
        "min_component_size": 500,
        "contour_filter": True,
        "contour_area_threshold": 3000,
        "line_mode": "centerline",
        "width_mode": "median_sampled_width",
    },
}

METHOD_ORDER = [
    "unet",
    "classifier_unet",
    "unet_postprocess_asta_none",
    "unet_postprocess_asta_median_sampled_width",
    "unet_postprocess_centerline_median_sampled_width",
    "classifier_unet_postprocess_asta_none",
    "classifier_unet_postprocess_asta_median_sampled_width",
    "classifier_unet_postprocess_centerline_median_sampled_width",
]

TITLES = {
    "image": "Image",
    "mask": "True Mask",
    "unet": "UNet",
    "classifier_unet": "UNet + Classifier",
    "unet_postprocess_asta_none": "UNet + ASTA",
    "unet_postprocess_asta_median_sampled_width": "UNet + ASTA Median Width",
    "unet_postprocess_centerline_median_sampled_width": "UNet + Centerline Median Width",
    "classifier_unet_postprocess_asta_none": "Classifier-UNet + ASTA",
    "classifier_unet_postprocess_asta_median_sampled_width": "Classifier-UNet + ASTA Median Width",
    "classifier_unet_postprocess_centerline_median_sampled_width": "Classifier-UNet + Centerline Median Width",
}

## Loading, Prediction, And Plot Helpers

In [ ]:
_MASK_CACHE = {}
_PREPROCESSING_CACHE = {}
_PREDICTION_CACHE = {}


def image_path(image_name):
    path = DATA_DIR / image_name
    if not path.exists():
        matches = sorted(DATA_DIR.glob("*.fits_full.png")) if DATA_DIR.exists() else []
        preview = "\n".join(f"  - {match.name}" for match in matches[:10])
        raise FileNotFoundError(
            f"Image not found: {path}\n"
            f"DATA_DIR is currently: {DATA_DIR}\n"
            f"Found {len(matches)} candidate image(s).\n{preview}"
        )
    return path


def mask_path(image_name):
    expected_name = image_name.replace(".fits_full.png", "_mask.png")
    path = DATA_DIR / expected_name
    if path.exists():
        return path

    if MASTER_SPLIT_CSV.exists():
        with MASTER_SPLIT_CSV.open(newline="") as file:
            for row in csv.reader(file):
                if len(row) >= 2 and row[0] == image_name:
                    path = DATA_DIR / row[1]
                    if path.exists():
                        return path

    matches = sorted(DATA_DIR.glob("*_mask.png")) if DATA_DIR.exists() else []
    preview = "\n".join(f"  - {match.name}" for match in matches[:10])
    raise FileNotFoundError(
        f"Mask not found for {image_name}. Expected: {DATA_DIR / expected_name}\n"
        f"DATA_DIR is currently: {DATA_DIR}\n"
        f"Found {len(matches)} candidate mask(s).\n{preview}"
    )


def selected_image_names():
    if SELECTED_IMAGE_NAMES:
        return list(SELECTED_IMAGE_NAMES)

    names = []
    if MASTER_SPLIT_CSV.exists():
        with MASTER_SPLIT_CSV.open(newline="") as file:
            for row in csv.reader(file):
                if len(row) >= 3 and row[2].strip() == str(SPLIT_ID) and (DATA_DIR / row[0]).exists():
                    names.append(row[0])
    if names:
        return names

    return [path.name for path in sorted(DATA_DIR.glob("*.fits_full.png"))]


def preprocessing_for(image_name):
    if image_name not in _PREPROCESSING_CACHE:
        _PREPROCESSING_CACHE[image_name] = pipeline.preprocessing(
            image_path(image_name),
            normalization=NORMALIZATION,
        )
    return _PREPROCESSING_CACHE[image_name]


def original_image(image_name):
    return preprocessing_for(image_name)["image"]


def true_mask(image_name):
    if image_name not in _MASK_CACHE:
        with Image.open(mask_path(image_name)) as image:
            mask = np.asarray(image)
            if mask.ndim == 3:
                mask = mask[..., 0]
            _MASK_CACHE[image_name] = (mask > 0).astype(np.uint8)
    return _MASK_CACHE[image_name]


def _base_prediction(image_name, *, use_classifier):
    preprocessing = preprocessing_for(image_name)
    prediction_data, _ = pipeline.segmentation(
        preprocessing["patch_data"],
        use_classifier=use_classifier,
        unet_threshold=UNET_THRESHOLD,
        classifier_threshold=CLASSIFIER_THRESHOLD,
    )
    return (prediction_data["segmented_result"] > 0).astype(np.uint8)


def _postprocess(mask, config):
    result = postprocess_segmentation(mask, **config)
    if isinstance(result, tuple):
        result = result[0]
    return (result > 0).astype(np.uint8)


def _postprocess_source(method_name):
    if method_name.startswith("classifier_unet_"):
        return "classifier_unet", method_name.removeprefix("classifier_unet_")
    if method_name.startswith("unet_"):
        return "unet", method_name.removeprefix("unet_")
    return None, None


def prediction_mask(image_name, method_name):
    key = (image_name, method_name)
    if key in _PREDICTION_CACHE:
        return _PREDICTION_CACHE[key]

    if method_name == "unet":
        mask = _base_prediction(image_name, use_classifier=False)
    elif method_name == "classifier_unet":
        mask = _base_prediction(image_name, use_classifier=True)
    else:
        base_method, postprocess_method = _postprocess_source(method_name)
        if postprocess_method not in POSTPROCESS_METHODS:
            raise ValueError(f"Unknown method: {method_name}")
        mask = _postprocess(prediction_mask(image_name, base_method), POSTPROCESS_METHODS[postprocess_method])

    mask = (mask > 0).astype(np.uint8)
    _PREDICTION_CACHE[key] = mask
    return mask


def clear_image_cache(image_name):
    for key in list(_PREDICTION_CACHE):
        if key[0] == image_name:
            del _PREDICTION_CACHE[key]
    _PREPROCESSING_CACHE.pop(image_name, None)
    _MASK_CACHE.pop(image_name, None)



def safe_filename(image_name):
    return re.sub(r"[^A-Za-z0-9_.-]+", "_", Path(image_name).stem)


def plot_all_pipelines(image_name, save=True, show=True):
    panels = [("image", original_image(image_name)), ("mask", true_mask(image_name))]
    panels.extend((method, prediction_mask(image_name, method)) for method in METHOD_ORDER)

    ncols = 5
    nrows = int(np.ceil(len(panels) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(4.2 * ncols, 4.2 * nrows), constrained_layout=True)
    axes = np.asarray(axes).reshape(-1)

    for ax, (name, image) in zip(axes, panels):
        ax.imshow(image, cmap="gray", vmin=0 if name != "image" else None, vmax=1 if name != "image" else None)
        ax.set_title(TITLES[name], fontsize=10)
        ax.axis("off")

    for ax in axes[len(panels):]:
        ax.axis("off")

    fig.suptitle(image_name, fontsize=14)

    save_path = None
    if save:
        OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
        save_path = OUTPUT_DIR / f"{safe_filename(image_name)}_all_pipelines.png"
        fig.savefig(save_path, dpi=180, bbox_inches="tight")

    if show:
        plt.show()
    else:
        plt.close(fig)

    clear_image_cache(image_name)
    return save_path, fig

## Preview One Image

In [ ]:
image_names = selected_image_names()
if not image_names:
    raise FileNotFoundError(f"No .fits_full.png images found in {DATA_DIR}")

print(f"Found {len(image_names)} image(s) to plot")
preview_path, _ = plot_all_pipelines(image_names[0], save=True, show=True)
print(f"Saved preview: {preview_path}")

## Save All Pipeline Plots

In [ ]:
saved_paths = []
for index, image_name in enumerate(image_names, start=1):
    print(f"[{index}/{len(image_names)}] Plotting {image_name}")
    save_path, _ = plot_all_pipelines(image_name, save=True, show=False)
    saved_paths.append(save_path)

print(f"Saved {len(saved_paths)} figure(s) to {OUTPUT_DIR}")